# 🎨 TRELLIS via Hugging Face (Easiest Method!)

Use the official TRELLIS Hugging Face Space to convert images to 3D.

**No GPU needed!** This runs on Hugging Face's servers.

---

## 📦 Step 1: Install Gradio Client

In [ ]:
print("Installing Gradio client...\n")
!pip install -q gradio_client
print("✅ Ready!")

## 🔗 Step 2: Connect to TRELLIS Space

In [ ]:
from gradio_client import Client, handle_file
from pathlib import Path

print("Connecting to TRELLIS Hugging Face Space...\n")

client = Client("JeffreyXiang/TRELLIS")

print("✅ Connected!")
print("\nSpace: https://huggingface.co/spaces/JeffreyXiang/TRELLIS")

## 📤 Step 3: Upload Your Image

In [ ]:
from google.colab import files
import shutil

# Create directory
input_dir = Path("/content/inputs")
input_dir.mkdir(exist_ok=True)

print("📤 Upload your image:\n")
uploaded = files.upload()

if uploaded:
    print("\n✅ Uploaded:")
    for name in uploaded:
        dst = input_dir / name
        Path(name).rename(dst)
        size_mb = dst.stat().st_size / 1e6
        print(f"   • {name} ({size_mb:.2f} MB)")
else:
    print("\n⚠️ No files uploaded")

## 🎨 Step 4: Convert to 3D!

This sends your image to Hugging Face's servers for processing.

In [ ]:
from IPython.display import display, Image as IPImage, FileLink
import matplotlib.pyplot as plt
from PIL import Image

# Get uploaded image
images = list(Path("/content/inputs").glob("*"))
images = [img for img in images if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

if not images:
    print("❌ No images found. Upload in Step 3.")
else:
    image_path = images[0]
    
    print(f"{'='*70}")
    print(f"Converting: {image_path.name}")
    print(f"{'='*70}\n")
    
    # Show preview
    img = Image.open(image_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Input: {image_path.name}")
    plt.tight_layout()
    plt.show()
    
    print(f"\n🚀 Sending to Hugging Face Space...")
    print("⏱️ This takes ~30-60 seconds...\n")
    
    try:
        # Call the TRELLIS API
        result = client.predict(
            image=handle_file(str(image_path)),
            seed=1,  # Change for different variations
            api_name="/image_to_3d"
        )
        
        print("✅ Processing complete!\n")
        
        # Result is a dictionary with GLB file path
        if isinstance(result, dict) and 'glb' in result:
            glb_path = result['glb']
        elif isinstance(result, str):
            glb_path = result
        elif isinstance(result, tuple):
            glb_path = result[0]
        else:
            print(f"⚠️ Unexpected result format: {type(result)}")
            print(f"Result: {result}")
            glb_path = None
        
        if glb_path:
            # Download the GLB file
            output_path = "/content/output.glb"
            
            # Copy from temp location
            import shutil
            shutil.copy(glb_path, output_path)
            
            size_mb = Path(output_path).stat().st_size / 1e6
            
            print(f"{'='*70}")
            print("✅ SUCCESS!")
            print(f"{'='*70}")
            print(f"\nFile: {output_path}")
            print(f"Size: {size_mb:.2f} MB\n")
            
            print("📥 Download your 3D model:")
            display(FileLink(output_path))
            
            print("\n💡 View at: https://gltf-viewer.donmccurdy.com")
        else:
            print("❌ Could not retrieve GLB file")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\n💡 Try using the web interface instead:")
        print("   https://huggingface.co/spaces/JeffreyXiang/TRELLIS")

## 🔄 Step 5: Batch Convert (Optional)

Convert multiple images at once.

In [ ]:
from pathlib import Path
import shutil

# Get all uploaded images
images = list(Path("/content/inputs").glob("*"))
images = [img for img in images if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]

if not images:
    print("❌ No images found")
else:
    output_dir = Path("/content/outputs")
    output_dir.mkdir(exist_ok=True)
    
    print(f"Converting {len(images)} images...\n")
    
    results = []
    
    for i, img_path in enumerate(images, 1):
        print(f"\n[{i}/{len(images)}] {img_path.name}")
        print("-" * 50)
        
        try:
            # Process image
            result = client.predict(
                image=handle_file(str(img_path)),
                seed=1,
                api_name="/image_to_3d"
            )
            
            # Get GLB path
            if isinstance(result, dict) and 'glb' in result:
                glb_path = result['glb']
            elif isinstance(result, str):
                glb_path = result
            elif isinstance(result, tuple):
                glb_path = result[0]
            else:
                glb_path = None
            
            if glb_path:
                # Save to output directory
                output_name = f"{img_path.stem}.glb"
                output_path = output_dir / output_name
                shutil.copy(glb_path, output_path)
                
                size_mb = output_path.stat().st_size / 1e6
                print(f"✅ Saved: {output_name} ({size_mb:.2f} MB)")
                results.append(output_path)
            else:
                print("❌ Failed to get GLB")
                
        except Exception as e:
            print(f"❌ Error: {e}")
            continue
    
    print(f"\n{'='*70}")
    print(f"✅ Batch complete: {len(results)}/{len(images)} successful")
    print(f"{'='*70}\n")
    
    if results:
        print("📥 Download your models:\n")
        for glb in sorted(output_dir.glob("*.glb")):
            size_mb = glb.stat().st_size / 1e6
            print(f"   {glb.name} ({size_mb:.2f} MB)")
            display(FileLink(str(glb)))

## 📦 Step 6: Download All as ZIP

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

output_dir = Path("/content/outputs")
glbs = list(output_dir.glob("*.glb"))

if glbs:
    print(f"📦 Creating zip with {len(glbs)} models...")
    
    archive = shutil.make_archive("/content/3d_models", 'zip', output_dir)
    size_mb = Path(archive).stat().st_size / 1e6
    
    print(f"   ✓ Archive: 3d_models.zip ({size_mb:.2f} MB)")
    print("\n📥 Downloading...")
    
    files.download(archive)
    print("\n✅ Download complete!")
else:
    print("❌ No GLB files found. Run Step 5 first.")

---

## 💡 About This Method

### Advantages
✅ **No GPU needed** - Runs on HF servers  
✅ **No installation hassles** - Just gradio_client  
✅ **Always up-to-date** - Uses official Space  
✅ **More reliable** - Maintained by TRELLIS team  

### How It Works
1. You upload image to Colab
2. Colab sends it to HF Space
3. HF Space processes with TRELLIS
4. Result downloads back to Colab
5. You download from Colab

### Alternative: Direct Web Interface
Even easier - just use the website directly:  
**https://huggingface.co/spaces/JeffreyXiang/TRELLIS**

No Colab needed at all!

---

## 🎯 View Your Models

- **[gltf-viewer.donmccurdy.com](https://gltf-viewer.donmccurdy.com)** - Drag & drop viewer
- [Babylon.js Sandbox](https://sandbox.babylonjs.com)
- Import in Blender (File → Import → glTF 2.0)
- Use in Unity, Unreal, Three.js

---

## 📚 Resources

- **TRELLIS Space**: https://huggingface.co/spaces/JeffreyXiang/TRELLIS
- **TRELLIS Model**: https://huggingface.co/JeffreyXiang/TRELLIS-image-large
- **GitHub**: https://github.com/microsoft/TRELLIS

---

## 🎉 Enjoy Creating 3D Models!

**This is the easiest way to use TRELLIS!** ✨